# NammaSpace — Dense Reconstruction on Colab (COLMAP + 3DGS)

Turns a capture video into a **dense Gaussian-splat** scene using the Colab T4 GPU
(what the 4 GB laptop can't do). Output is a `.ply` you download and compress to
`.ksplat` for the browser viewer.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`. Then run cells top to bottom.
Put your capture video in Google Drive first, e.g. `MyDrive/nammaspace/apartment.mp4`.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (inputs + outputs persist here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Settings  — edit these two lines

In [ ]:
import os
VIDEO    = '/content/drive/MyDrive/nammaspace/apartment.mp4'   # <-- your capture in Drive
SCENE_ID = 'apartment'                                          # <-- name for this scene
WORK = '/content/ns'
os.makedirs(WORK, exist_ok=True)
assert os.path.exists(VIDEO), 'Video not found - check the Drive path above'
print('video OK:', VIDEO)

## 4. Install COLMAP + Nerfstudio (gsplat-based 3DGS)
Takes ~8-12 min the first time. Nerfstudio bundles frame extraction, COLMAP SfM,
`splatfacto` (gsplat 3DGS training) and splat export.

In [ ]:
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg
!pip -q install nerfstudio
print('\ninstalled. colmap:', end=' '); !colmap -h 2>/dev/null | head -1

## 5. Video -> frames -> camera poses (COLMAP)
`ns-process-data` samples frames, de-duplicates and runs COLMAP. `--num-frames-target`
~300 keeps overlap without overloading SfM. (Use more for a whole apartment.)

In [ ]:
!ns-process-data video --data "$VIDEO" \
    --output-dir "$WORK/$SCENE_ID" \
    --num-frames-target 300 \
    --sfm-tool colmap --matching-method sequential

## 6. Train the dense Gaussian splat (splatfacto)
30k steps on the T4. `cull-alpha-thresh` prunes low-opacity **floaters**
(raise it toward 0.1-0.15 if the result is hazy).

In [ ]:
!ns-train splatfacto \
    --data "$WORK/$SCENE_ID" \
    --output-dir "$WORK/outputs" \
    --max-num-iterations 30000 \
    --pipeline.model.cull-alpha-thresh 0.1 \
    --viewer.quit-on-train-completion True

## 7. Export the splat as .ply

In [ ]:
import glob
cfg = sorted(glob.glob(f'{WORK}/outputs/**/config.yml', recursive=True))[-1]
print('using config:', cfg)
!ns-export gaussian-splat --load-config "$cfg" --output-dir "$WORK/export"
!ls -lh $WORK/export

## 8. Save the result to Drive (so it survives disconnect)

In [ ]:
out = f'/content/drive/MyDrive/nammaspace/out/{SCENE_ID}'
os.makedirs(out, exist_ok=True)
!cp $WORK/export/*.ply "$out/"
# also save the COLMAP poses (needed for the scene contract: cameras.json / metric scale)
!cp -r $WORK/$SCENE_ID/colmap/sparse "$out/" 2>/dev/null; cp -r $WORK/$SCENE_ID/sparse "$out/" 2>/dev/null
print('saved to Drive:', out); !ls -lh "$out"

## 9. Back on your laptop
Download `out/<scene>/*.ply` from Drive, then compress + deploy:
```bash
node viewer/convert-ksplat.mjs <downloaded>.ply viewer/public/scenes/<scene>/splat/scene.ksplat
```
Then build `scene.json` (and `cameras.json` from the saved COLMAP `sparse/`) per
`docs/round2-scene-contract.md`, and view with `?scene=<scene>`.

**Floater / quality tips:** raise `cull-alpha-thresh` (step 6); capture per the SOP
(sharp frames, parallax); more frames -> denser multi-room coverage.